In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
print(f'Q1)    {len(documents)} lesson pages are in the dataset')

Q1)    72 lesson pages are in the dataset


In [4]:
from minsearch import Index
index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
q2_results = index.search(query)
print(f'Q2)     The filename of the first result is \'{q2_results[0]['filename']}\'')

Q2)     The filename of the first result is '01-agentic-rag/lessons/14-agentic-loop.md'


In [5]:
from dotenv import load_dotenv
load_dotenv()

from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)


In [6]:
answerQ3 = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answerQ3)

('The loop keeps calling the model by checking whether the response contains any `function_call` items.\n\n- If there is a function call, the code runs the tool, appends the tool result to `messages`, and loops again.\n- If there are no function calls, it breaks out of the loop and stops.\n\nSo the stop condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nIn other words, it keeps going until the model returns a final message with no more tool calls.', ResponseUsage(input_tokens=7135, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=108, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7243))


In [7]:
print(f'Q3)    We have sent {answerQ3[1].input_tokens} input tokens')

Q3)    We have sent 7135 input tokens


In [8]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [9]:
print(f'Q4)    We get {len(chunks)} chunks')

Q4)    We get 295 chunks


In [10]:
index.fit(chunks)
answerQ5 = assistant.rag("How does the agentic loop keep calling the model until it stops?")
ratio = int(answerQ3[1].input_tokens / answerQ5[1].input_tokens)
print(f'Q5)    We have sent {answerQ5[1].input_tokens} input tokens , {ratio} times fewer')

Q5)    We have sent 2318 input tokens , 3 times fewer


In [11]:
from toyaikit.tools import Tools
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback
from toyaikit.llm import OpenAIClient

In [12]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
""".strip()

In [13]:
def search(query):
    """
    Search the lessons for entries matching the given query.
    """
    boost_dict = {'content': 3.0, 'filename': 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [14]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [15]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [16]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


In [17]:
counts = 0

for m in result.all_messages:
    t = m["type"] if isinstance(m, dict) else m.type
    if t == "function_call":
        counts += 1

print(f'Q6)    Agent has called {counts} times \'search\'')

Q6)    Agent has called 3 times 'search'
